In [ ]:
from google.colab import drive
import os
import shutil

# 1. Google Drive'ı Bağla
drive.mount('/content/drive')

# 2. Drive'dan yedeklenen dosyaları yerel disk alanına kopyala
print("Yedek dosyaları Google Drive'dan indiriliyor...")
shutil.copy('/content/drive/My Drive/opencv_cuda/cv2_cuda_backup.zip', '/content/')
shutil.copy('/content/drive/My Drive/opencv_cuda/opencv_libs_backup.zip', '/content/')

# 3. Python paket klasörünü oluştur ve dosyaları çıkart
print("Python cv2 paketi kuruluyor...")
os.makedirs('/usr/local/lib/python3.13/dist-packages/cv2', exist_ok=True)
!unzip -q -o /content/cv2_cuda_backup.zip -d /usr/local/lib/python3.13/dist-packages/cv2

# 4. Paylaşılan C++ kütüphanelerini çıkart
print("Sistem paylaşımlı kütüphaneleri (.so) geri yükleniyor...")
!unzip -q -o /content/opencv_libs_backup.zip -d /usr/local/lib

# 5. Sistem dinamik kütüphane bağlayıcısını güncelle
print("Kütüphane yolları sistemde güncelleniyor...")
!ldconfig

# 6. Test Et
print("\n=== KURULUM TAMAMLANDI ===")
import cv2
cuda_devices = cv2.cuda.getCudaEnabledDeviceCount()
print(f"CUDA Destekli Cihaz Sayısı: {cuda_devices}")
if cuda_devices > 0:
    print("Tebrikler! OpenCV CUDA desteğiyle başarıyla geri yüklendi.")
    cv2.cuda.printCudaDeviceInfo(0)
else:
    print("Uyarı: Kurulum tamamlandı ancak GPU etkinleştirilmemiş olabilir. Runtime ayarlarından GPU seçtiğinize emin olun.")

Mounted at /content/drive
Yedek dosyaları Google Drive'dan indiriliyor...
Python cv2 paketi kuruluyor...
Sistem paylaşımlı kütüphaneleri (.so) geri yükleniyor...
Kütüphane yolları sistemde güncelleniyor...
/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_5.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libumf.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbb.so.12 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero_v2.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc_proxy.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libhwloc.so.15 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_opencl.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libjulia.so.1.12 is not a sym

In [6]:
import cv2
import numpy as np

input_path = "resim1_8k.bmp"

# OpenCV resmi BGR olarak açar
bgr = cv2.imread(input_path, cv2.IMREAD_COLOR)

if bgr is None:
    raise FileNotFoundError(f"{input_path} bulunamadı.")

# BGR'den RGB'ye dönüştür
rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

# HSV dönüşümü
hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)

# YUV dönüşümü
yuv = cv2.cvtColor(rgb, cv2.COLOR_RGB2YUV)

# CIE Lab dönüşümü
lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2Lab)

# HSI dönüşümü (OpenCV'de doğrudan HSI dönüşümü yok)
rgb_float = rgb.astype(np.float32) / 255.0
r = rgb_float[:, :, 0]
g = rgb_float[:, :, 1]
b = rgb_float[:, :, 2]

total = r + g + b
minimum = np.minimum(np.minimum(r, g), b)

saturation = np.zeros_like(total)
np.divide(3.0 * minimum, total, out=saturation, where=total > 1e-6)
saturation = 1.0 - saturation
saturation[total <= 1e-6] = 0

numerator = 0.5 * ((r - g) + (r - b))
denominator = np.sqrt((r - g) ** 2 + (r - b) * (g - b))

cos_theta = np.ones_like(total)
np.divide(numerator, denominator, out=cos_theta, where=denominator > 1e-6)
theta = np.arccos(np.clip(cos_theta, -1.0, 1.0))

hue = np.where(b <= g, theta, 2.0 * np.pi - theta)
hue[denominator <= 1e-6] = 0

intensity = total / 3.0

# H, S ve I kanallarını 0–255 aralığına getir
hsi = np.stack([
    hue / (2.0 * np.pi) * 255.0,
    saturation * 255.0,
    intensity * 255.0
], axis=2).clip(0, 255).astype(np.uint8)

# RGB renk kanallarını BMP'ye uygun BGR sırasıyla kaydet
def save_rgb_channels_as_bmp(filename, data):
    data_bgr = cv2.cvtColor(data, cv2.COLOR_RGB2BGR)
    if not cv2.imwrite(filename, data_bgr):
        raise IOError(f"{filename} kaydedilemedi.")

save_rgb_channels_as_bmp("resim_HSV.bmp", hsv)
save_rgb_channels_as_bmp("resim_HSI.bmp", hsi)
save_rgb_channels_as_bmp("resim_YUV.bmp", yuv)
save_rgb_channels_as_bmp("resim_Lab.bmp", lab)

print("Dönüştürülen görüntüler BMP formatında kaydedildi.")

Dönüştürülen görüntüler BMP formatında kaydedildi.
